# RAGSec Guard - Day 2: fine-tune + evaluate (Kaggle, free GPU)

**What this does:** fine-tunes Qwen2.5-7B-Instruct (QLoRA) on your 5,200-row
guard dataset, then scores the **base model vs your tuned model** on the
915-row held-out test set and prints the comparison table - the numbers that
go on the resume.

**Before you run (one-time, in Kaggle):**
1. Account verified by phone (Settings - Phone verification) - unlocks GPU + internet.
2. This notebook's Settings: **Accelerator = GPU T4 x2**, **Internet = ON**.
3. Add the data: create a Kaggle Dataset with `guard_train.jsonl` and
   `guard_test.jsonl`, then attach it to this notebook (Add Data). The next
   cells find the files automatically anywhere under `/kaggle/input`.
4. Use **Save & Run All** so it keeps running in the background (free quota:
   ~30 GPU-hours/week; this job needs roughly 2-4 hours).

**Outputs** (in the notebook's Output tab when finished): the LoRA adapter
folder, `eval_results.json`, and the printed base-vs-tuned table.
Download them - Kaggle working storage is not permanent.

In [ ]:
# 0. Check the GPU you got
!nvidia-smi

In [ ]:
# 1. Install Unsloth + training stack (Kaggle recipe)
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Locate the data files (attached Kaggle dataset, anywhere under /kaggle/input)
import os

def find_file(name):
    for root, _dirs, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    for root, _dirs, files in os.walk("/kaggle/working"):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(f"{name} not found - attach the dataset with the two JSONL files first.")

TRAIN_PATH = find_file("guard_train.jsonl")
TEST_PATH = find_file("guard_test.jsonl")
print("train:", TRAIN_PATH)
print("test: ", TEST_PATH)

In [ ]:
# 3. Config - the only knobs you might touch
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # pre-quantized, fits one T4 (16 GB)
EPOCHS = 2
MAX_SEQ_LEN = 1024
EVAL_LIMIT = 0          # 0 = full 915-row test set; set e.g. 200 for a quick smoke run
OUTPUT_DIR = "/kaggle/working/ragsec-guard-adapter"
SEED = 42

In [ ]:
# 4. Load the base model (4-bit) and attach LoRA
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("model ready")

In [ ]:
# 5. Build the training text (Qwen chat format: user asks, assistant labels)
import json
from datasets import Dataset

def load_rows(path):
    return [json.loads(line) for line in open(path)]

def to_text(row):
    messages = [
        {"role": "user", "content": row["instruction"] + "\n\n" + row["input"]},
        {"role": "assistant", "content": row["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False)}

train_rows = load_rows(TRAIN_PATH)
train_ds = Dataset.from_list([to_text(r) for r in train_rows])
print(f"training rows: {len(train_ds)}")

In [ ]:
# 6. Train (QLoRA SFT) - roughly 1.5-3 hours on one T4 for 2 epochs
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,      # effective batch 8
        warmup_steps=10,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        logging_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir="/kaggle/working/training-output",
        report_to="none",
    ),
)
trainer.train()
print("training done")

In [ ]:
# 7. Save the adapter (your model = base + this small folder)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("adapter saved to", OUTPUT_DIR)

## Evaluation - base vs tuned on the held-out test set

Runs the tuned model first (already in memory), then reloads the plain base
model and runs the same prompts. Parses the label from the first output line
and computes accuracy, per-class precision/recall/F1, and the false-positive
rate on safe prompts (the number that decides whether a guard is usable).

In [ ]:
# 8. Evaluation helpers
import gc
import json
import re
from collections import Counter

LABELS = ["safe", "injection", "jailbreak"]
test_rows = load_rows(TEST_PATH)
if EVAL_LIMIT:
    test_rows = test_rows[:EVAL_LIMIT]
print(f"evaluating on {len(test_rows)} held-out rows")

def build_prompt(row):
    messages = [{"role": "user",
                 "content": row["instruction"] + "\n\n" + row["input"]}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)

def parse_label(text):
    first = text.strip().splitlines()[0].strip().lower() if text.strip() else ""
    for lab in LABELS:
        if first.startswith(lab):
            return lab
    m = re.search(r"\b(safe|injection|jailbreak)\b", text.lower())
    return m.group(1) if m else "unparsed"

@torch.no_grad()
def run_eval(mdl, batch_size=16):
    FastLanguageModel.for_inference(mdl)
    preds = []
    prompts = [build_prompt(r) for r in test_rows]
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=MAX_SEQ_LEN).to("cuda")
        out = mdl.generate(**inputs, max_new_tokens=48, do_sample=False,
                           pad_token_id=tokenizer.pad_token_id)
        for j in range(len(batch)):
            gen = out[j][inputs["input_ids"].shape[1]:]
            preds.append(parse_label(tokenizer.decode(gen, skip_special_tokens=True)))
        if (i // batch_size) % 10 == 0:
            print(f"  {i + len(batch)}/{len(prompts)}")
    return preds

def score(preds):
    gold = [r["label"] for r in test_rows]
    acc = sum(p == g for p, g in zip(preds, gold)) / len(gold)
    per = {}
    for lab in LABELS:
        tp = sum(p == lab and g == lab for p, g in zip(preds, gold))
        fp = sum(p == lab and g != lab for p, g in zip(preds, gold))
        fn = sum(p != lab and g == lab for p, g in zip(preds, gold))
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        per[lab] = {"precision": round(prec, 3), "recall": round(rec, 3),
                    "f1": round(2 * prec * rec / (prec + rec), 3) if prec + rec else 0.0}
    safe_fp = (sum(p != "safe" and g == "safe" for p, g in zip(preds, gold))
               / max(1, sum(g == "safe" for g in gold)))
    return {"accuracy": round(acc, 4), "per_class": per,
            "false_positive_rate_on_safe": round(safe_fp, 4),
            "confusion": {f"{g} -> {p}": n for (g, p), n in Counter((g, p) for g, p in zip(gold, preds)).items()}}

In [ ]:
# 9. Evaluate the TUNED model
tuned_preds = run_eval(model)
tuned_scores = score(tuned_preds)
print("tuned:", tuned_scores["accuracy"])

In [ ]:
# 10. Free memory, load the BASE model, evaluate it the same way
del model, trainer
gc.collect()
torch.cuda.empty_cache()

base_model, _ = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)
base_preds = run_eval(base_model)
base_scores = score(base_preds)
print("base:", base_scores["accuracy"])

In [ ]:
# 11. The table that matters + save everything
results = {"base": base_scores, "tuned": tuned_scores,
           "test_rows": len(test_rows), "train_rows": len(train_rows),
           "base_model": BASE_MODEL, "epochs": EPOCHS}
with open("/kaggle/working/eval_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"{'metric':<28}{'base':>10}{'tuned':>10}")
print(f"{'accuracy':<28}{base_scores['accuracy']:>10}{tuned_scores['accuracy']:>10}")
for lab in LABELS:
    print(f"{'F1 ' + lab:<28}{base_scores['per_class'][lab]['f1']:>10}"
          f"{tuned_scores['per_class'][lab]['f1']:>10}")
print(f"{'false alarms on safe':<28}"
      f"{base_scores['false_positive_rate_on_safe']:>10}"
      f"{tuned_scores['false_positive_rate_on_safe']:>10}")
print("\nsaved: adapter ->", OUTPUT_DIR, "| results -> /kaggle/working/eval_results.json")

## After it finishes

1. Download from the Output tab: the `ragsec-guard-adapter` folder and
   `eval_results.json`. Send both back to Milo (or drop them in the project
   folder) - Day 3 turns them into the demo, the GitHub repo, and the resume
   bullet with the real numbers filled in.
2. Optional: merge the adapter into a full model later on a bigger GPU.
   Not needed for the eval story - the adapter + base model name is the model.